# 🤖 Building AI Agents with Gemini: From One Decision to an Autonomous Loop

### Dinesh AI Academy | Day 4 — Agents & MCP

**Learning objective:**
By the end of this notebook you will be able to explain, and build from scratch,
a working AI agent — no framework, no magic, just Gemini + a Python loop.

**Where we left off (Day 3):** we built one round trip — ask Gemini, it optionally
requests **one** tool, we run it, we send the result back, Gemini answers. That
round trip only ever makes **one decision**.

**Where we're going today:** what if the goal needs *several* decisions in a row,
and we don't know in advance how many, or in what order? That's an **agent**.

## 1. What Is an Agent, Exactly?

> **An agent is a loop.** At every turn, the LLM looks at the goal and everything
> that has happened so far, and decides the *next* action — call a tool, or stop
> and answer. Your application keeps that loop running until the LLM decides
> it's done (or a safety limit is hit).

Compare the two mental models:

```text
Day 3 — a single tool-calling round trip (one decision):

   User question
        |
      Gemini  -- decides once --> tool call OR final answer
        |
   (if tool)  run it, send result back, Gemini answers
        |
      Done.


Day 4 — an agent (many decisions, repeated until done):

        +----------------------------------------+
        |                                         |
   Goal -> Gemini decides: tool, or final answer?  |
        |        |                     |           |
        |     tool call            final answer    |
        |        |                     |           |
        |   run the tool               v           |
        |        |                   Done.         |
        |   observe result                         |
        |        |                                 |
        +--------+   (loop back to "Gemini decides" again)
```

The only structural difference between Day 3 and today is: **we wrap the round
trip in a loop**, and keep feeding every tool result back in, so Gemini can
decide to call *another* tool based on what it just learned — or stop.

| Term | Definition | Who decides the next step? |
|---|---|---|
| **Tool** | A single capability (calculator, weather lookup, database query) | — |
| **Workflow** | A fixed sequence of steps *you* wrote in code | The developer, in advance |
| **Agent** | A loop where the model picks the next action based on the goal + what happened so far | The model, at runtime |

**One sentence to remember:** *Tool = capability. Workflow = a script. Agent = a script that lets the model choose its own next line.*

## 2. Setup — Gemini API Key

Same pattern as Day 3: works locally (via `.env`) or in Google Colab (via Secrets),
without changing any code.

**Never publish your API key in a notebook, GitHub repository, Moodle, WhatsApp group, or screenshot.**

> **Free tier note:** a free Gemini API key allows only a few requests per
> minute. A single agent run can make several requests in a row (one per
> loop step), so if you run every cell quickly you may briefly see a
> `RESOURCE_EXHAUSTED` (429) error. The agent loop below automatically waits
> and retries once -- if it still fails, just wait ~30 seconds and re-run
> the cell.

In [1]:
# Install the current Google GenAI Python SDK.
# In Google Colab, run this cell once.

!pip -q install -U google-genai


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from google import genai
from google.genai import types
import os

def get_secret(key_name: str) -> str:
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except ImportError:
        from dotenv import load_dotenv, find_dotenv
        load_dotenv(find_dotenv())
        return os.getenv(key_name)

GAISTUDIO_API_KEY = get_secret("GAISTUDIO_API_KEY")

if not GAISTUDIO_API_KEY:
    raise ValueError(
        "GAISTUDIO_API_KEY not found. Set it in Colab Secrets or in your local .env file."
    )

print("API key loaded successfully.")

client = genai.Client(api_key=GAISTUDIO_API_KEY)

# You can change this model if your account has access to another Gemini model.
MODEL = "gemini-3.5-flash-lite"

print("Gemini client is ready.")
print("Model:", MODEL)

API key loaded successfully.
Gemini client is ready.
Model: gemini-3.5-flash-lite


## 3. Give the Agent Some Tools

Three plain Python functions — nothing Gemini-specific about them. Two are
carried over from Day 3 for continuity; `get_weather` is new and is what will
let us build a genuine **multi-step** goal further down (a question whose answer
needs the output of one tool as the *input* to another).

`get_weather` is intentionally **simulated** (a small fixed lookup table, no
internet call) — same idea as the Day 3 Streamlit demo: free, offline, and
100% reproducible for a classroom, with zero API keys beyond Gemini's own.

In [9]:
from datetime import datetime
from zoneinfo import ZoneInfo

def calculate(a: float, b: float, operation: str) -> float:
    """Perform a basic arithmetic calculation."""
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Cannot divide by zero.")
        return a / b
    else:
        raise ValueError(f"Unsupported operation: {operation}")


def get_current_time(timezone: str) -> dict:
    """Get the current date and time for an IANA timezone such as Asia/Tokyo."""
    now = datetime.now(ZoneInfo(timezone))
    return {
        "timezone": timezone,
        "date": now.strftime("%Y-%m-%d"),
        "time": now.strftime("%H:%M:%S"),
        "formatted": now.strftime("%A, %d %B %Y at %I:%M:%S %p")
    }


# A tiny, fixed, offline weather lookup -- simulated on purpose (see note above).
WEATHER_DB = {
    "tokyo": {"temp_c": 26, "condition": "sunny"},
    "paris": {"temp_c": 18, "condition": "cloudy"},
    "mumbai": {"temp_c": 31, "condition": "humid, partly cloudy"},
    "new york": {"temp_c": 21, "condition": "rainy"},
    "london": {"temp_c": 16, "condition": "overcast"},
}

def get_weather(city: str) -> dict:
    """Get the current simulated weather for a city (demo data, not a live API)."""
    data = WEATHER_DB.get(city.strip().lower())
    if data is None:
        return {"city": city, "temp_c": 20, "condition": "unknown", "note": "city not in demo dataset"}
    return {"city": city, **data}


# Quick sanity check -- call each function directly, no Gemini involved yet.
print("Calculator:", calculate(25, 40, "multiply"))
print("Current time:", get_current_time("Asia/Kolkata"))
print("Weather:", get_weather("Tokyo"))

Calculator: 1000
Current time: {'timezone': 'Asia/Kolkata', 'date': '2026-09-17', 'time': '19:48:43', 'formatted': 'Thursday, 17 September 2026 at 07:48:43 PM'}
Weather: {'city': 'Tokyo', 'temp_c': 26, 'condition': 'sunny'}


## 4. Describe the Tools to Gemini

Same idea as Day 3: Gemini can only see this JSON menu, never your Python
source. We also add a `TOOLBOX` dict mapping each tool's name to the *actual*
function -- that's how our own code will turn "Gemini asked for `get_weather`"
back into a real function call in a moment.

In [10]:
calculator_declaration = {
    "name": "calculate",
    "description": "Performs basic arithmetic calculations.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "The first number."},
            "b": {"type": "number", "description": "The second number."},
            "operation": {
                "type": "string",
                "description": "The arithmetic operation.",
                "enum": ["add", "subtract", "multiply", "divide"]
            }
        },
        "required": ["a", "b", "operation"]
    }
}

time_declaration = {
    "name": "get_current_time",
    "description": "Gets the current date and time for an IANA timezone such as Asia/Kolkata or Asia/Tokyo.",
    "parameters": {
        "type": "object",
        "properties": {
            "timezone": {
                "type": "string",
                "description": "IANA timezone name, for example Asia/Kolkata, Asia/Tokyo, or America/New_York."
            }
        },
        "required": ["timezone"]
    }
}

weather_declaration = {
    "name": "get_weather",
    "description": "Gets the current simulated weather (temperature in Celsius, condition) for a city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name, e.g. 'Tokyo'."}
        },
        "required": ["city"]
    }
}

# Maps a tool NAME (a string Gemini sends us) to the REAL function that runs it.
# This is the one place our code bridges "what the model asked for" to
# "what actually executes" -- see Section 6.
TOOLBOX = {
    "calculate": calculate,
    "get_current_time": get_current_time,
    "get_weather": get_weather,
}

tools = types.Tool(
    function_declarations=[
        calculator_declaration,
        time_declaration,
        weather_declaration,
    ]
)

config = types.GenerateContentConfig(
    tools=[tools],
    system_instruction=(
        "You are a helpful assistant that can use tools to answer questions "
        "accurately. Use get_weather for weather questions, get_current_time "
        "for current time/date questions, and calculate for arithmetic where "
        "accuracy matters. If a question needs the result of one tool as input "
        "to another (for example, converting a temperature you just looked up), "
        "call the tools one after another rather than guessing the second value. "
        "If no tool is needed, answer directly."
    )
)

print("Three tools are available to Gemini:", list(TOOLBOX.keys()))

Three tools are available to Gemini: ['calculate', 'get_current_time', 'get_weather']


# 🎯 5. Why a Single Round Trip Isn't Enough

Try a goal that needs **two tools, where the second one depends on the first's result**:

> "What's the current temperature in Tokyo in Celsius, and what is that in Fahrenheit?"

Gemini can't answer the Fahrenheit part without first knowing the Celsius value --
and it doesn't know that until `get_weather` actually runs. Watch what happens
with only **one** round trip (ask -> maybe one tool -> answer), exactly like Day 3:

In [11]:
user_prompt = "What's the current temperature in Tokyo in Celsius, and what is that in Fahrenheit?"

response = client.models.generate_content(
    model=MODEL,
    contents=user_prompt,
    config=config
)

for part in response.candidates[0].content.parts:
    if part.function_call:
        print("Gemini requested:", part.function_call.name, dict(part.function_call.args))
    if part.text:
        print("Gemini text:", part.text)

Gemini requested: get_weather {'city': 'Tokyo'}


Notice: Gemini can only request **one step at a time**. After this single round
trip, it has asked for the weather -- but it hasn't converted anything to
Fahrenheit yet, because it doesn't have the Celsius number in front of it yet.

If we stop here (like Day 3 did), the conversation is unfinished. We'd have to
manually: run the tool, send the result back, check if Gemini asks for
*another* tool, run that too, send it back again... That's exactly the
repetition a loop is for.

## 6. Build the Agent Loop

This is the entire idea of an agent, in code. Compare it to the diagram in
Section 1 -- it's the *same* four steps (ask -> decide -> act -> observe), just
wrapped in a loop that keeps going until Gemini stops asking for tools:

```text
for step in range(max_steps):
    1. Ask Gemini, with the full conversation so far
    2. If Gemini's reply has NO function_call -> done, return the text
    3. Otherwise: run the requested tool(s) ourselves
    4. Append the tool result(s) to the conversation
    5. Loop back to step 1 -- Gemini now sees the result and decides again
```

A `max_steps` safety cap exists because a model can occasionally get stuck
re-requesting the same tool instead of answering -- we never want an agent that
can loop forever (or run up an unbounded API bill).

In [12]:
import time

def execute_tool(name: str, args: dict):
    """Turn a tool name + arguments (from Gemini) into a real function call."""
    func = TOOLBOX.get(name)
    if func is None:
        raise ValueError(f"Unknown tool requested: {name}")
    return func(**args)


def call_gemini(contents, retries: int = 5, backoff_seconds: float = 20.0):
    """
    generate_content() with a tiny retry loop for transient 429 rate-limit
    errors -- the Gemini free tier allows only a few requests per minute,
    and a multi-step agent can easily exceed that during a classroom demo.
    A production system would use a proper backoff library; this is the
    minimum viable version so the notebook stays reliable on the free tier.
    """
    for attempt in range(1, retries + 1):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            if "RESOURCE_EXHAUSTED" in str(e) and attempt < retries:
                print(f"Rate limited, waiting {backoff_seconds:.0f}s before retry {attempt}/{retries - 1}...")
                time.sleep(backoff_seconds)
            else:
                raise


def run_agent(goal: str, max_steps: int = 5, verbose: bool = True) -> str:
    """
    The agent loop: repeatedly ask Gemini, execute whatever tool it requests,
    and feed the result back -- until it answers with plain text, or we hit
    max_steps.
    """
    # `contents` is the agent's growing memory of this run: the goal, every
    # tool-call request Gemini made, and every tool result we sent back.
    contents = [
        types.Content(role="user", parts=[types.Part.from_text(text=goal)])
    ]

    if verbose:
        print(f"GOAL: {goal}\n{'-' * 60}")

    for step in range(1, max_steps + 1):
        response = call_gemini(contents)

        # Keep Gemini's own turn in the conversation before we look at it,
        # so the next request includes exactly what Gemini said this time.
        model_turn = response.candidates[0].content
        contents.append(model_turn)

        function_calls = [p.function_call for p in model_turn.parts if p.function_call]

        if not function_calls:
            # No tool requested this turn -- Gemini considers itself done.
            final_text = response.text or ""
            if verbose:
                print(f"Step {step}: Gemini answered directly (no tool needed).")
                print(f"\nFINAL ANSWER:\n{final_text}")
            return final_text

        # Run every tool Gemini asked for this turn, and collect the results
        # as a single user turn (Gemini can request more than one tool at once).
        result_parts = []
        for fc in function_calls:
            args = dict(fc.args)
            if verbose:
                print(f"Step {step}: Gemini requested tool `{fc.name}` with {args}")
            try:
                result = execute_tool(fc.name, args)
            except Exception as e:
                result = {"error": str(e)}
            if verbose:
                print(f"         -> result: {result}")
            response_payload = result if isinstance(result, dict) else {"result": result}
            result_parts.append(
                types.Part.from_function_response(name=fc.name, response=response_payload)
            )

        contents.append(types.Content(role="user", parts=result_parts))

    # Safety net: the loop ran out of steps without Gemini producing a final answer.
    if verbose:
        print(f"Stopped after {max_steps} steps without a final answer.")
    return "(agent hit the step limit before finishing)"

print("Agent loop is ready.")

Agent loop is ready.


## 7. Run It — the Two-Tool, Chained Goal

Same question as Section 5, but this time through `run_agent()`. Watch the
printed trace: Gemini should call `get_weather` first, see the Celsius value,
*then* call `calculate` to convert it -- two decisions, made one after another,
each informed by what came before.

In [13]:
run_agent("What's the current temperature in Tokyo in Celsius, and what is that in Fahrenheit?")

GOAL: What's the current temperature in Tokyo in Celsius, and what is that in Fahrenheit?
------------------------------------------------------------
Step 1: Gemini requested tool `get_weather` with {'city': 'Tokyo'}
         -> result: {'city': 'Tokyo', 'temp_c': 26, 'condition': 'sunny'}
Step 2: Gemini requested tool `calculate` with {'a': 26, 'b': 1.8, 'operation': 'multiply'}
         -> result: 46.800000000000004
Step 3: Gemini requested tool `calculate` with {'b': 32, 'operation': 'add', 'a': 46.8}
         -> result: 78.8
Step 4: Gemini answered directly (no tool needed).

FINAL ANSWER:
The current temperature in Tokyo is 26°C, which is 78.8°F.


'The current temperature in Tokyo is 26°C, which is 78.8°F.'

## 8. Run It — a Few More Goals

Try goals that need a different number of steps, so you can see the loop
adapt: zero tools, one tool, and two *independent* tools in one go.

In [14]:
# No tool needed at all -- Gemini should answer directly on step 1.
run_agent("In one sentence, what is an AI agent?")

GOAL: In one sentence, what is an AI agent?
------------------------------------------------------------
Step 1: Gemini answered directly (no tool needed).

FINAL ANSWER:
An AI agent is an autonomous software program that perceives its environment through sensors, makes decisions using artificial intelligence, and takes actions through actuators to achieve specific goals.


'An AI agent is an autonomous software program that perceives its environment through sensors, makes decisions using artificial intelligence, and takes actions through actuators to achieve specific goals.'

In [15]:
# Exactly one tool.
run_agent("What time is it right now in Tokyo?")

GOAL: What time is it right now in Tokyo?
------------------------------------------------------------
Step 1: Gemini requested tool `get_current_time` with {'timezone': 'Asia/Tokyo'}
         -> result: {'timezone': 'Asia/Tokyo', 'date': '2026-09-17', 'time': '23:19:27', 'formatted': 'Thursday, 17 September 2026 at 11:19:27 PM'}
Step 2: Gemini answered directly (no tool needed).

FINAL ANSWER:
It is currently Thursday, September 17, 2026, at 11:19 PM in Tokyo.


'It is currently Thursday, September 17, 2026, at 11:19 PM in Tokyo.'

In [16]:
# Two tools, but independent of each other (order doesn't matter, unlike Section 7).
run_agent("What's the weather in Paris, and what time is it there right now?")

GOAL: What's the weather in Paris, and what time is it there right now?
------------------------------------------------------------
Step 1: Gemini requested tool `get_weather` with {'city': 'Paris'}
         -> result: {'city': 'Paris', 'temp_c': 18, 'condition': 'cloudy'}
Step 1: Gemini requested tool `get_current_time` with {'timezone': 'Europe/Paris'}
         -> result: {'timezone': 'Europe/Paris', 'date': '2026-09-17', 'time': '16:19:32', 'formatted': 'Thursday, 17 September 2026 at 04:19:32 PM'}
Step 2: Gemini answered directly (no tool needed).

FINAL ANSWER:
The current weather in Paris is cloudy with a temperature of 18°C. The current time there is 4:19 PM on Thursday, 17 September 2026.


'The current weather in Paris is cloudy with a temperature of 18°C. The current time there is 4:19 PM on Thursday, 17 September 2026.'

# 🧠 9. What Actually Makes This an *Agent* (and Not Just a Longer Workflow)?

Look closely at `run_agent()`: notice what our Python code does **not** decide.

- We never wrote `if "weather" in goal: call get_weather()`.
- We never hardcoded "always call the weather tool before the calculator."
- We never told it how many steps a particular goal would take.

**Gemini decided all of that, at runtime, from the goal text alone** -- which
tool(s) to call, in what order, whether a second tool was even needed, and
when to stop. Our code only supplied the *capabilities* (the tools) and the
*loop* (keep going until done). That division of responsibility is the whole
definition of an agent:

| | Workflow | Agent |
|---|---|---|
| Who picks the next step? | The developer, hardcoded in advance | The model, at runtime |
| Can it skip a step it doesn't need? | No -- the code always runs it | Yes -- it just won't call that tool |
| Can it handle a goal you didn't anticipate? | Only if you wrote a branch for it | Often yes, if the right tools exist |
| Predictability | High | Lower -- needs guardrails (next section) |

# 🛡️ 10. Production Safety Note — Agents Need *More* Guardrails Than a Single Tool Call

An agent that can take multiple, self-chosen steps is more powerful -- and more
dangerous -- than a single tool call. Everything from Day 3's safety note still
applies, plus:

- **`max_steps` (we used 5)** -- without it, a confused model can loop
  indefinitely, burning API quota/cost with every extra step.
- **Cost awareness** -- every step in the loop is a *full* LLM call. A 5-step
  agent run costs roughly 5x a single request. Log step counts in production.
- **Tool allow-lists** -- `TOOLBOX` only contains what we explicitly added.
  Never let a model call an arbitrary function by name.
- **Idempotency for risky tools** -- if a tool sends an email or charges a card,
  a retried step could run it twice. Read-only tools like ours are safe to
  repeat; side-effecting ones need extra care.
- **Human approval before high-impact actions** -- exactly like Day 3: the loop
  can *request* an action, but your application should still gate anything
  irreversible (payments, deletions, sending messages) behind a real check.
- **Timeouts per step** -- a single slow tool call (e.g. a hanging network
  request) can stall the entire agent. Add per-tool timeouts in real systems.

```text
Agent requests: send_email(to="someone@example.com")
                              |
                 Your application checks:
                 Is this user allowed? Is this within the step budget?
                 Does policy require human approval first?
                              |
                     Execute / Reject
```

# 🧪 11. Classroom Challenge

For each goal below, predict **before running it**: how many tool calls will
the agent make, and in what order? Then try it and compare.

| Goal | Your prediction |
|---|---|
| "Convert 45°C to Fahrenheit." | ? |
| "What's 18% of 2400, and what time is it in London?" | ? |
| "Is it warmer in Mumbai or Paris right now?" | ? |
| "What is the capital of France?" | ? |

The last one is a trick question for a reason -- **no tool we built can answer
"capital of France," and none is needed.** A well-built agent recognizes that
too, and just answers from its own knowledge instead of forcing a tool call.

In [17]:
run_agent("Is it warmer in Mumbai or Paris right now?")

GOAL: Is it warmer in Mumbai or Paris right now?
------------------------------------------------------------
Step 1: Gemini requested tool `get_weather` with {'city': 'Mumbai'}
         -> result: {'city': 'Mumbai', 'temp_c': 31, 'condition': 'humid, partly cloudy'}
Step 1: Gemini requested tool `get_weather` with {'city': 'Paris'}
         -> result: {'city': 'Paris', 'temp_c': 18, 'condition': 'cloudy'}
Step 2: Gemini answered directly (no tool needed).

FINAL ANSWER:
It is currently warmer in Mumbai. Mumbai is at 31°C, while Paris is at 18°C.


'It is currently warmer in Mumbai. Mumbai is at 31°C, while Paris is at 18°C.'

## 🌉 Next: MCP — Reusable Tools You Didn't Have to Build

Every tool in this notebook was something **we** wrote by hand: the Python
function *and* its JSON schema description. That's fine for three toy tools --
but real agents often need access to dozens of tools built by other teams
entirely (a company database, a filesystem, Slack, GitHub, a CRM), and
hand-writing + maintaining a schema for each one doesn't scale.

**MCP (Model Context Protocol)** is a standard that lets an agent *discover*
and *call* tools exposed by any MCP-compatible server, without you writing
custom integration code for each one -- the server describes its own tools in
a standard format, and any MCP-aware agent (built with any SDK, in any
language) can use them immediately.

That's the next notebook: connecting the agent loop you just built to tools
you *didn't* have to write yourself.

# 🎓 Day 4 Takeaway

By the end of this notebook, you should be able to explain:

1. What an agent is, in one sentence, and how it differs from a single
   tool-calling round trip and from a hardcoded workflow.
2. Why a fixed number of tool calls isn't enough for goals whose steps
   depend on each other.
3. The five parts of the agent loop: ask -> decide -> act -> observe -> repeat.
4. Why `max_steps` (or an equivalent safety cap) is not optional in a real
   agent.
5. What makes something an *agent* rather than a workflow: **who** decides
   the next step, and **when**.
6. Why agents need stronger safety guardrails than a single tool call.

### The complete mental model

```text
        Goal
          |
   +------v------+
   |   Gemini    |<----------------+
   |  (decide)   |                 |
   +------+------+                 |
          |                        |
   tool needed?  -- no --> Final answer
          | yes                    |
   +------v------+                 |
   | Run the real |                |
   | Python tool  |                |
   +------+------+                 |
          |                        |
     Observe result ---------------+
     (feed back in, loop again)
```

## Official references

- Gemini API — Function calling: https://ai.google.dev/gemini-api/docs/function-calling
- Gemini API — Getting started: https://ai.google.dev/gemini-api/docs/get-started
- Gemini API — Tools: https://ai.google.dev/gemini-api/docs/tools
- Model Context Protocol (MCP) — Introduction: https://modelcontextprotocol.io/introduction
- Google AI Studio: https://aistudio.google.com/

This notebook intentionally builds the agent loop manually (no agent framework)
so every decision point is visible. Frameworks like LangChain/LangGraph, the
OpenAI Agents SDK, and Google's ADK automate exactly this loop -- now that
you've built one by hand, you'll recognize the same five steps inside any of
them.